# NovaCart — Silver Order Reviews Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 2. Define Storage Paths

In [0]:
BRONZE_ORDER_REVIEWS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/order_reviews"
)

SILVER_ORDER_REVIEWS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/order_reviews"
)

QUARANTINE_ORDER_REVIEWS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/order_reviews"
)

print(f"Bronze path: {BRONZE_ORDER_REVIEWS_PATH}")
print(f"Silver path: {SILVER_ORDER_REVIEWS_PATH}")
print(f"Quarantine path: {QUARANTINE_ORDER_REVIEWS_PATH}")

## 3. Read Bronze Order Reviews Data

In [0]:
order_reviews_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_ORDER_REVIEWS_PATH)
)

bronze_row_count = order_reviews_bronze_df.count()

print("Bronze order reviews loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

order_reviews_bronze_df.printSchema()
display(order_reviews_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in order_reviews_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Review Scores

In [0]:
display(
    order_reviews_bronze_df
    .groupBy("review_score")
    .count()
    .orderBy("review_score")
)

## 6. Profile Missing and Invalid Values

In [0]:
order_reviews_profile_df = order_reviews_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("review_id").isNull()
            | (F.trim(F.col("review_id")) == "")
        ).cast("int")
    ).alias("invalid_review_id"),

    F.sum(
        (
            F.col("order_id").isNull()
            | (F.trim(F.col("order_id")) == "")
        ).cast("int")
    ).alias("invalid_order_id"),

    F.sum(
        (
            F.col("review_score").isNull()
            | (F.col("review_score") < 1)
            | (F.col("review_score") > 5)
        ).cast("int")
    ).alias("invalid_review_score"),

    F.sum(
        F.col("review_creation_date").isNull().cast("int")
    ).alias("missing_review_creation_date"),

    F.sum(
        F.col("review_answer_timestamp").isNull().cast("int")
    ).alias("missing_review_answer_timestamp"),

    F.sum(
        F.col("review_comment_title").isNull().cast("int")
    ).alias("missing_review_comment_title"),

    F.sum(
        F.col("review_comment_message").isNull().cast("int")
    ).alias("missing_review_comment_message"),

    F.sum(
        (
            F.col("review_answer_timestamp").isNotNull()
            & F.col("review_creation_date").isNotNull()
            & (
                F.col("review_answer_timestamp")
                < F.col("review_creation_date")
            )
        ).cast("int")
    ).alias("answer_before_creation")
)

display(order_reviews_profile_df)

## 7. Print Full Review Quality Profile

In [0]:
profile = order_reviews_profile_df.first().asDict()

for metric, value in profile.items():
    print(f"{metric}: {value}")

## 8. Check Duplicate Review IDs

In [0]:
duplicate_review_ids_df = (
    order_reviews_bronze_df
    .groupBy("review_id")
    .count()
    .filter(
        F.col("review_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_review_id_count = duplicate_review_ids_df.count()

print(
    f"Number of review_id values appearing more than once: "
    f"{duplicate_review_id_count}"
)

display(duplicate_review_ids_df.limit(20))

## 9. Check Duplicate Review-Order Keys

In [0]:
duplicate_review_order_keys_df = (
    order_reviews_bronze_df
    .groupBy(
        "review_id",
        "order_id"
    )
    .count()
    .filter(
        F.col("review_id").isNotNull()
        & F.col("order_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_review_order_key_count = (
    duplicate_review_order_keys_df.count()
)

print(
    "Number of duplicate "
    "(review_id, order_id) keys: "
    f"{duplicate_review_order_key_count}"
)

display(duplicate_review_order_keys_df.limit(20))

## 10. Check Exact Duplicate Records

In [0]:
business_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
]

exact_duplicate_count = (
    bronze_row_count
    - order_reviews_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate review rows: {exact_duplicate_count}")

## 11. Clean and Standardize Review Fields

In [0]:
order_reviews_cleaned_df = (
    order_reviews_bronze_df
    .withColumn(
        "review_id",
        F.trim(F.col("review_id"))
    )
    .withColumn(
        "order_id",
        F.trim(F.col("order_id"))
    )
    .withColumn(
        "review_comment_title",
        F.when(
            F.col("review_comment_title").isNull(),
            F.lit(None)
        ).otherwise(
            F.trim(F.col("review_comment_title"))
        )
    )
    .withColumn(
        "review_comment_message",
        F.when(
            F.col("review_comment_message").isNull(),
            F.lit(None)
        ).otherwise(
            F.trim(F.col("review_comment_message"))
        )
    )
)

## 12. Convert Blank Comments to Null

In [0]:
order_reviews_cleaned_df = (
    order_reviews_cleaned_df
    .withColumn(
        "review_comment_title",
        F.when(
            F.col("review_comment_title") == "",
            F.lit(None)
        ).otherwise(F.col("review_comment_title"))
    )
    .withColumn(
        "review_comment_message",
        F.when(
            F.col("review_comment_message") == "",
            F.lit(None)
        ).otherwise(F.col("review_comment_message"))
    )
)
review_key_window = Window.partitionBy(
    "review_id",
    "order_id"
)

order_reviews_checked_df = (
    order_reviews_cleaned_df
    .withColumn(
        "_duplicate_key_count",
        F.count("*").over(review_key_window)
    )
)

## 13. Define Review Validation Rules

In [0]:
invalid_review_id_condition = (
    F.col("review_id").isNull()
    | (F.col("review_id") == "")
)

invalid_order_id_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")
)

invalid_review_score_condition = (
    F.col("review_score").isNull()
    | (F.col("review_score") < 1)
    | (F.col("review_score") > 5)
)

missing_review_creation_date_condition = (
    F.col("review_creation_date").isNull()
)

missing_review_answer_timestamp_condition = (
    F.col("review_answer_timestamp").isNull()
)

answer_before_creation_condition = (
    F.col("review_answer_timestamp").isNotNull()
    & F.col("review_creation_date").isNotNull()
    & (
        F.col("review_answer_timestamp")
        < F.col("review_creation_date")
    )
)

## 14. Assign Review Rejection Reasons

In [0]:
order_reviews_validated_df = order_reviews_checked_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_review_id_condition,
        F.lit("MISSING_REVIEW_ID")
    )
    .when(
        invalid_order_id_condition,
        F.lit("MISSING_ORDER_ID")
    )
    .when(
        invalid_review_score_condition,
        F.lit("INVALID_REVIEW_SCORE")
    )
    .when(
        missing_review_creation_date_condition,
        F.lit("MISSING_REVIEW_CREATION_DATE")
    )
    .when(
        missing_review_answer_timestamp_condition,
        F.lit("MISSING_REVIEW_ANSWER_TIMESTAMP")
    )
    .when(
        answer_before_creation_condition,
        F.lit("ANSWER_BEFORE_CREATION")
    )
    .when(
        F.col("_duplicate_key_count") > 1,
        F.lit("DUPLICATE_REVIEW_ORDER_KEY")
    )
    .otherwise(F.lit(None))
)

## 15. Review Validation Results

In [0]:
display(
    order_reviews_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 16. Split Valid and Invalid Reviews

In [0]:
order_reviews_valid_df = (
    order_reviews_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop(
        "_rejection_reason",
        "_duplicate_key_count"
    )
)

order_reviews_quarantine_df = (
    order_reviews_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
    .drop("_duplicate_key_count")
)

## 17. Add Silver Processing Metadata

In [0]:
order_reviews_silver_df = (
    order_reviews_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 18. Add Quarantine Metadata

In [0]:
order_reviews_quarantine_df = (
    order_reviews_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("order_reviews")
    )
)

## 19. Count Silver and Quarantine Records

In [0]:
valid_row_count = order_reviews_silver_df.count()
quarantine_row_count = order_reviews_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 20. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 21. Write Valid Reviews to Silver

In [0]:
(
    order_reviews_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_ORDER_REVIEWS_PATH)
)

print("Silver order reviews written successfully.")

## 22. Write Invalid Reviews to Quarantine

In [0]:
(
    order_reviews_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_ORDER_REVIEWS_PATH)
)

print("Order reviews quarantine output written successfully.")

## 23. Read Written Delta Outputs

In [0]:
order_reviews_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_ORDER_REVIEWS_PATH)
)

order_reviews_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_ORDER_REVIEWS_PATH)
)

silver_written_count = order_reviews_silver_written_df.count()
quarantine_written_count = order_reviews_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 24. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver order reviews pipeline completed successfully.")
print("Final row-count validation passed.")

## 25. Inspect Final Silver Order Reviews Dataset

In [0]:
order_reviews_silver_written_df.printSchema()

display(
    order_reviews_silver_written_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp",
        "_silver_processed_at"
    ).limit(20)
)